## 1. 환경 설정

In [2]:
import sys
import json
import asyncio
from pathlib import Path
from datetime import datetime

# 프로젝트 루트 경로 설정
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"프로젝트 루트: {project_root}")
print(f"현재 작업 디렉토리: {Path.cwd()}")

프로젝트 루트: /Users/jaehyeokchoi/Desktop/chois_toy/private/TableMagnifier
현재 작업 디렉토리: /Users/jaehyeokchoi/Desktop/chois_toy/private/TableMagnifier/QA_example


In [11]:
# QA Generator 모듈 임포트 (변경 시 리로드)
import importlib
import QA_example.qa_generator
importlib.reload(QA_example.qa_generator)

from QA_example.qa_generator import (
    InsuranceTableQAGenerator,
    QADifficulty,
    QAType,
    generate_qa_from_tables,
)

# Gemini API Pool 임포트
from polling_gemini import get_gemini_pool

print("✅ 모듈 임포트 완료 (리로드됨)")

✅ 모듈 임포트 완료 (리로드됨)


In [12]:
# API Pool 상태 확인
pool = get_gemini_pool()
print("📊 API Pool 상태:")
print(f"  현재 키: {pool.get_current_key_info()['name']}")
print(f"  총 키 수: {pool.get_current_key_info()['total_keys']}")

📊 API Pool 상태:
  현재 키: key1
  총 키 수: 3


## 2. 샘플 테이블 데이터 준비

보험 도메인의 다양한 테이블 예시를 준비합니다.

In [13]:
# 샘플 테이블 1: 보험료 산출 기초율 테이블 (실제 데이터 기반)
TABLE_1_PREMIUM_CALCULATION = """
|구분|XX세|XX+1세|XX+2세|XX+3세|XX+4세|XX+5세|
|---|---|---|---|---|---|---|
|나이증가분(A)||1059|1357|1739|2229|2855|
|보험료 산출 기초율(위험률 등) 증가분(B=전년도 기준보험료의 최대 25% 가정)||10846|13897|17806|22815|29232|
|기준보험료(C=전년도 기준보험료+A+B)|42325|54321|69485|89030|114074|146161|
""".strip()

# 샘플 테이블 2: 해지환급금 예시표
TABLE_2_SURRENDER_VALUE = """
|경과기간|납입보험료 누계|해지환급금|환급률|
|---|---|---|---|
|1년|600000|0|0%|
|3년|1800000|540000|30%|
|5년|3000000|1650000|55%|
|10년|6000000|4800000|80%|
|15년|9000000|8550000|95%|
|20년(만기)|12000000|12000000|100%|
""".strip()

# 샘플 테이블 3: 보장 내역표
TABLE_3_COVERAGE = """
|보장항목|보장내용|지급금액|지급조건|
|---|---|---|---|
|사망보험금|일반사망|50000000|피보험자 사망시|
|사망보험금|재해사망|100000000|재해로 인한 사망시|
|암진단금|일반암|30000000|암 최초 진단시|
|암진단금|소액암|6000000|소액암 진단시|
|암진단금|유사암|3000000|유사암 진단시|
|입원비|일반입원|50000|1일당 (최대 180일)|
|입원비|암입원|100000|1일당 (최대 180일)|
|수술비|일반수술|500000|1회당|
|수술비|암수술|2000000|1회당|
""".strip()

# 샘플 테이블 4: 연령별 보험료 예시
TABLE_4_AGE_PREMIUM = """
|가입연령|성별|20년납_월보험료|전기납_월보험료|일시납_총보험료|
|---|---|---|---|---|
|30세|남|45000|38000|8500000|
|30세|여|42000|35000|7800000|
|40세|남|65000|52000|11500000|
|40세|여|58000|47000|10200000|
|50세|남|95000|75000|16000000|
|50세|여|82000|65000|14000000|
""".strip()

print("✅ 샘플 테이블 데이터 준비 완료")
print(f"  - TABLE_1: 보험료 산출 기초율")
print(f"  - TABLE_2: 해지환급금 예시")
print(f"  - TABLE_3: 보장 내역")
print(f"  - TABLE_4: 연령별 보험료")

✅ 샘플 테이블 데이터 준비 완료
  - TABLE_1: 보험료 산출 기초율
  - TABLE_2: 해지환급금 예시
  - TABLE_3: 보장 내역
  - TABLE_4: 연령별 보험료


In [14]:
# 테이블 딕셔너리 구성
tables = {
    "table_1_premium_calculation": TABLE_1_PREMIUM_CALCULATION,
    "table_2_surrender_value": TABLE_2_SURRENDER_VALUE,
    "table_3_coverage": TABLE_3_COVERAGE,
    "table_4_age_premium": TABLE_4_AGE_PREMIUM,
}

# 테이블 미리보기
from IPython.display import display, Markdown

for table_id, content in tables.items():
    print(f"\n{'='*60}")
    print(f"📋 {table_id}")
    print('='*60)
    display(Markdown(content))


📋 table_1_premium_calculation


|구분|XX세|XX+1세|XX+2세|XX+3세|XX+4세|XX+5세|
|---|---|---|---|---|---|---|
|나이증가분(A)||1059|1357|1739|2229|2855|
|보험료 산출 기초율(위험률 등) 증가분(B=전년도 기준보험료의 최대 25% 가정)||10846|13897|17806|22815|29232|
|기준보험료(C=전년도 기준보험료+A+B)|42325|54321|69485|89030|114074|146161|


📋 table_2_surrender_value


|경과기간|납입보험료 누계|해지환급금|환급률|
|---|---|---|---|
|1년|600000|0|0%|
|3년|1800000|540000|30%|
|5년|3000000|1650000|55%|
|10년|6000000|4800000|80%|
|15년|9000000|8550000|95%|
|20년(만기)|12000000|12000000|100%|


📋 table_3_coverage


|보장항목|보장내용|지급금액|지급조건|
|---|---|---|---|
|사망보험금|일반사망|50000000|피보험자 사망시|
|사망보험금|재해사망|100000000|재해로 인한 사망시|
|암진단금|일반암|30000000|암 최초 진단시|
|암진단금|소액암|6000000|소액암 진단시|
|암진단금|유사암|3000000|유사암 진단시|
|입원비|일반입원|50000|1일당 (최대 180일)|
|입원비|암입원|100000|1일당 (최대 180일)|
|수술비|일반수술|500000|1회당|
|수술비|암수술|2000000|1회당|


📋 table_4_age_premium


|가입연령|성별|20년납_월보험료|전기납_월보험료|일시납_총보험료|
|---|---|---|---|---|
|30세|남|45000|38000|8500000|
|30세|여|42000|35000|7800000|
|40세|남|65000|52000|11500000|
|40세|여|58000|47000|10200000|
|50세|남|95000|75000|16000000|
|50세|여|82000|65000|14000000|

## 3. QA Generator 초기화

In [15]:
# QA Generator 인스턴스 생성
qa_generator = InsuranceTableQAGenerator(
    model_name="gemini-2.0-flash"  # 또는 "gemini-1.5-flash"
)

print("✅ QA Generator 초기화 완료")
print(f"  사용 모델: gemini-2.0-flash")

✅ QA Generator 초기화 완료
  사용 모델: gemini-2.0-flash


## 4. 난이도별 QA 생성

### 4.1 IR (Information Retrieval) - 단순 정보 검색

In [16]:
# IR 난이도 QA 생성 (단답형, 특정 셀 검색)
print("🔍 IR (Information Retrieval) QA 생성 중...")

ir_qa_pairs = qa_generator.generate_qa_by_difficulty(
    tables=tables,
    difficulty=QADifficulty.IR,
    num_questions=3
)

print(f"\n✅ {len(ir_qa_pairs)}개의 IR QA 생성 완료")
for qa in ir_qa_pairs:
    print(f"\n📌 [{qa.id}] {qa.difficulty}")
    print(f"   Q: {qa.question}")
    print(f"   A: {qa.answer}")
    if qa.evidence:
        print(f"   Evidence: {qa.evidence}")

🔍 IR (Information Retrieval) QA 생성 중...

✅ 3개의 IR QA 생성 완료

📌 [IR_001] IR
   Q: XX세의 기준보험료는 얼마인가요?
   A: 42325
   Evidence: {'table_id': 'table_1_premium_calculation', 'row': '기준보험료(C=전년도 기준보험료+A+B)', 'column': 'XX세'}

📌 [IR_002] IR
   Q: 경과기간이 1년일 때 해지환급금은 얼마인가요?
   A: 0
   Evidence: {'table_id': 'table_2_surrender_value', 'row': '1년', 'column': '해지환급금'}

📌 [IR_003] IR
   Q: 30세 남성의 20년납 월보험료는 얼마인가요?
   A: 45000
   Evidence: {'table_id': 'table_4_age_premium', 'row': '30세 (남)', 'column': '20년납_월보험료'}

✅ 3개의 IR QA 생성 완료

📌 [IR_001] IR
   Q: XX세의 기준보험료는 얼마인가요?
   A: 42325
   Evidence: {'table_id': 'table_1_premium_calculation', 'row': '기준보험료(C=전년도 기준보험료+A+B)', 'column': 'XX세'}

📌 [IR_002] IR
   Q: 경과기간이 1년일 때 해지환급금은 얼마인가요?
   A: 0
   Evidence: {'table_id': 'table_2_surrender_value', 'row': '1년', 'column': '해지환급금'}

📌 [IR_003] IR
   Q: 30세 남성의 20년납 월보험료는 얼마인가요?
   A: 45000
   Evidence: {'table_id': 'table_4_age_premium', 'row': '30세 (남)', 'column': '20년납_월보험료'}


### 4.2 Analysis - 분석적 질문

In [17]:
# Analysis 난이도 QA 생성
print("📊 Analysis QA 생성 중...")

analysis_qa_pairs = qa_generator.generate_qa_by_difficulty(
    tables=tables,
    difficulty=QADifficulty.ANALYSIS,
    num_questions=3
)

print(f"\n✅ {len(analysis_qa_pairs)}개의 Analysis QA 생성 완료")
for qa in analysis_qa_pairs:
    print(f"\n📌 [{qa.id}] {qa.difficulty}")
    print(f"   Q: {qa.question}")
    print(f"   A: {qa.answer}")
    if qa.reasoning:
        print(f"   Reasoning: {qa.reasoning}")

📊 Analysis QA 생성 중...

✅ 3개의 Analysis QA 생성 완료

📌 [ANALYSIS_001] Analysis
   Q: 암진단금 보장항목 중에서 가장 높은 지급금액을 제공하는 보장내용은 무엇인가요?
   A: 일반암
   Reasoning: table_3_coverage에서 '보장항목'이 '암진단금'인 행들을 확인하고, 그 중 '지급금액'이 가장 높은 '보장내용'을 찾습니다. 일반암은 30,000,000원, 소액암은 6,000,000원, 유사암은 3,000,000원이므로 일반암이 가장 높습니다.

📌 [ANALYSIS_002] Analysis
   Q: 해지환급률이 50% 이상이 되려면 최소 몇 년의 경과기간이 필요합니까?
   A: 5년
   Reasoning: table_2_surrender_value에서 '환급률'이 50% 이상인 행들을 찾고, 그 중 가장 작은 '경과기간'을 확인합니다. 3년 경과 시 환급률은 30%이고, 5년 경과 시 환급률은 55%이므로 최소 5년이 필요합니다.

📌 [ANALYSIS_003] Analysis
   Q: 40세 가입자의 경우, 남성과 여성 중 누가 20년납_월보험료가 더 높은가요?
   A: 남성
   Reasoning: table_4_age_premium에서 '가입연령'이 40세인 행들을 찾고, 해당 행들의 '20년납_월보험료'를 비교합니다. 40세 남성의 20년납_월보험료는 65,000원이고, 40세 여성의 20년납_월보험료는 58,000원이므로 남성이 더 높습니다.

✅ 3개의 Analysis QA 생성 완료

📌 [ANALYSIS_001] Analysis
   Q: 암진단금 보장항목 중에서 가장 높은 지급금액을 제공하는 보장내용은 무엇인가요?
   A: 일반암
   Reasoning: table_3_coverage에서 '보장항목'이 '암진단금'인 행들을 확인하고, 그 중 '지급금액'이 가장 높은 '보장내용'을 찾습니다. 일반암은 30,000,000원, 소액암은 6,000,000원, 유사암

### 4.3 Compare (Multi-hop) - 비교 및 다중 추론

In [18]:
# Compare 난이도 QA 생성 (Multi-hop 추론)
print("⚖️ Compare (Multi-hop) QA 생성 중...")

compare_qa_pairs = qa_generator.generate_qa_by_difficulty(
    tables=tables,
    difficulty=QADifficulty.COMPARE,
    num_questions=3
)

print(f"\n✅ {len(compare_qa_pairs)}개의 Compare QA 생성 완료")
for qa in compare_qa_pairs:
    print(f"\n📌 [{qa.id}] {qa.difficulty}")
    print(f"   Q: {qa.question}")
    print(f"   A: {qa.answer}")
    if qa.reasoning:
        print(f"   Reasoning: {qa.reasoning}")

⚖️ Compare (Multi-hop) QA 생성 중...

✅ 3개의 Compare QA 생성 완료

📌 [COMPARE_001] Compare
   Q: table_1_premium_calculation에서 XX+1세의 경우, 보험료 산출 기초율 증가분(B)은 나이증가분(A)보다 얼마나 더 큰가요?
   A: 9,787원 더 큽니다.
   Reasoning: XX+1세의 보험료 산출 기초율 증가분(B) 값에서 나이증가분(A) 값을 차감하여 그 차이를 계산합니다.

📌 [COMPARE_002] Compare
   Q: table_2_surrender_value에서 보험 가입 후 5년 경과 시점과 10년 경과 시점의 해지환급금은 얼마나 차이가 나나요?
   A: 3,150,000원 차이가 납니다.
   Reasoning: 10년 경과 시점의 해지환급금에서 5년 경과 시점의 해지환급금을 차감하여 차이를 계산합니다.

📌 [COMPARE_003] Compare
   Q: table_4_age_premium에서 30세 남성이 20년납 월보험료로 가입했을 경우, 20년 만기 시점의 총 납입보험료와 table_2_surrender_value의 20년 경과 시점 해지환급금은 얼마나 차이가 나나요?
   A: 1,200,000원 차이가 납니다.
   Reasoning: table_4에서 30세 남성의 20년납 월보험료를 찾아 20년간의 총 납입보험료를 계산하고, table_2에서 20년 경과 시점의 해지환급금을 찾아 두 값의 차이를 계산합니다.

✅ 3개의 Compare QA 생성 완료

📌 [COMPARE_001] Compare
   Q: table_1_premium_calculation에서 XX+1세의 경우, 보험료 산출 기초율 증가분(B)은 나이증가분(A)보다 얼마나 더 큰가요?
   A: 9,787원 더 큽니다.
   Reasoning: XX+1세의 보험료 산출 기초율 증가분(B) 값에서 나이증가분(A) 값을 차감하여 그 차이를 계산합니다.

📌 [COMPARE_

### 4.4 Aggregation - 집계 연산

In [19]:
# Aggregation 난이도 QA 생성 (수치 계산)
print("➕ Aggregation QA 생성 중...")

agg_qa_pairs = qa_generator.generate_qa_by_difficulty(
    tables=tables,
    difficulty=QADifficulty.AGGREGATION,
    num_questions=3
)

print(f"\n✅ {len(agg_qa_pairs)}개의 Aggregation QA 생성 완료")
for qa in agg_qa_pairs:
    print(f"\n📌 [{qa.id}] {qa.difficulty}")
    print(f"   Q: {qa.question}")
    print(f"   A: {qa.answer}")
    if qa.python_verification:
        print(f"   Python 검증 코드 포함: ✅")

➕ Aggregation QA 생성 중...

✅ 3개의 Aggregation QA 생성 완료

📌 [AGG_001] Aggregation
   Q: table_1_premium_calculation에서 XX+1세부터 XX+5세까지 '나이증가분(A)'의 총합은 얼마인가요?
   A: 9239
   Python 검증 코드 포함: ✅

📌 [AGG_002] Aggregation
   Q: table_2_surrender_value에서 경과기간 1년, 3년, 5년, 10년, 15년, 20년 시점의 '해지환급금'의 평균은 얼마인가요?
   A: 4590000
   Python 검증 코드 포함: ✅

📌 [AGG_003] Aggregation
   Q: table_4_age_premium에서 30세 남성이 20년납으로 가입했을 경우, 20년 동안 총 납입해야 할 보험료는 얼마인가요?
   A: 10800000
   Python 검증 코드 포함: ✅

✅ 3개의 Aggregation QA 생성 완료

📌 [AGG_001] Aggregation
   Q: table_1_premium_calculation에서 XX+1세부터 XX+5세까지 '나이증가분(A)'의 총합은 얼마인가요?
   A: 9239
   Python 검증 코드 포함: ✅

📌 [AGG_002] Aggregation
   Q: table_2_surrender_value에서 경과기간 1년, 3년, 5년, 10년, 15년, 20년 시점의 '해지환급금'의 평균은 얼마인가요?
   A: 4590000
   Python 검증 코드 포함: ✅

📌 [AGG_003] Aggregation
   Q: table_4_age_premium에서 30세 남성이 20년납으로 가입했을 경우, 20년 동안 총 납입해야 할 보험료는 얼마인가요?
   A: 10800000
   Python 검증 코드 포함: ✅


### 4.5 Reasoning - 복합 추론

In [20]:
# Reasoning 난이도 QA 생성 (Chain-of-Thought)
print("🧠 Reasoning QA 생성 중...")

reasoning_qa_pairs = qa_generator.generate_qa_by_difficulty(
    tables=tables,
    difficulty=QADifficulty.REASONING,
    num_questions=3
)

print(f"\n✅ {len(reasoning_qa_pairs)}개의 Reasoning QA 생성 완료")
for qa in reasoning_qa_pairs:
    print(f"\n📌 [{qa.id}] {qa.difficulty}")
    print(f"   Q: {qa.question}")
    print(f"   A: {qa.answer}")
    if qa.chain_of_thought:
        print(f"   Chain of Thought:")
        for i, step in enumerate(qa.chain_of_thought, 1):
            print(f"      Step {i}: {step}")

🧠 Reasoning QA 생성 중...

✅ 3개의 Reasoning QA 생성 완료

📌 [REASON_001] Reasoning
   Q: 30세 남성이 20년납 월보험료로 가입했을 때, `table_1`의 '기준보험료' 증가율이 '월보험료'에 동일하게 적용된다고 가정하면, 가입 후 3년이 지난 시점(즉, 33세)의 월보험료는 얼마가 될까요?
   A: 약 94,658원
   Chain of Thought:
      Step 1: Step 1: `table_4`에서 30세 남성의 20년납 월보험료를 확인합니다: 45,000원.
      Step 2: Step 2: `table_1`에서 'XX세'를 30세로 가정하고, 'XX세' (30세)의 기준보험료(C)는 42,325원, 'XX+3세' (33세)의 기준보험료(C)는 89,030원임을 확인합니다.
      Step 3: Step 3: 30세 대비 33세의 기준보험료 증가율을 계산합니다: (89,030 / 42,325).
      Step 4: Step 4: 이 증가율을 30세 남성의 초기 월보험료에 적용하여 33세의 월보험료를 계산합니다: 45,000원 * (89,030 / 42,325) = 94,657.76...
      Step 5: Step 5: 계산된 값을 반올림하여 최종 월보험료를 도출합니다.

📌 [REASON_002] Reasoning
   Q: 3년 동안 보험료를 납입한 계약자가 일반암 진단을 받고, 진단금을 수령한 직후 보험을 해지한다면, 이 계약자가 총 얼마의 금액을 받게 될까요? (단, 납입보험료는 해지환급금 계산에만 영향을 미치며, 진단금 수령 시점까지의 납입보험료는 해지환급금 누계에 포함된다고 가정합니다.)
   A: 30,540,000원
   Chain of Thought:
      Step 1: Step 1: `table_3`에서 '일반암' 진단금을 확인합니다: 30,000,000원.
      Step 2: Step 2: `table_2`에서 '경과기간' 3년 시점의

### 4.6 Insight - 통찰 도출

In [21]:
# Insight 난이도 QA 생성 (서술형)
print("💡 Insight QA 생성 중...")

insight_qa_pairs = qa_generator.generate_qa_by_difficulty(
    tables=tables,
    difficulty=QADifficulty.INSIGHT,
    num_questions=2
)

print(f"\n✅ {len(insight_qa_pairs)}개의 Insight QA 생성 완료")
for qa in insight_qa_pairs:
    print(f"\n📌 [{qa.id}] {qa.difficulty}")
    print(f"   Q: {qa.question}")
    print(f"   A: {qa.answer[:200]}..." if len(qa.answer) > 200 else f"   A: {qa.answer}")

💡 Insight QA 생성 중...

✅ 2개의 Insight QA 생성 완료

📌 [INSIGHT_001] Insight
   Q: 이 보험 상품의 해지환급금 데이터를 분석했을 때, 조기 해지가 고객에게 미치는 재정적 영향은 무엇이며, 이는 상품 설계 철학에 대해 어떤 시사점을 제공합니까?
   A: 이 보험 상품을 조기에 해지할 경우, 고객은 상당한 재정적 손실을 입게 됩니다. 예를 들어, 1년 경과 후 해지 시 납입보험료 누계의 0%만 환급되며, 3년 후에도 30%만 돌려받습니다. 납입한 원금(환급률 100%)을 온전히 돌려받으려면 20년 만기까지 유지해야 합니다. 이는 이 상품이 장기적인 보장과 유지를 전제로 설계되었으며, 단기 해지에 대해 높은 ...

📌 [INSIGHT_002] Insight
   Q: 보험료 산출 구성 요소와 연령 및 성별에 따른 월 보험료 데이터를 종합적으로 고려할 때, 이 보험 상품이 위험을 평가하고 보험료를 책정하는 방식에 대해 어떤 통찰을 얻을 수 있으며, 특히 연령이 증가함에 따라 위험 평가가 어떻게 변화하는 것으로 보입니까?
   A: 이 보험 상품은 연령이 증가함에 따라 위험 관련 요소를 보험료 산출에 매우 중요하게 반영하고 있습니다. `table_1`에서 '나이증가분(A)'보다 '보험료 산출 기초율(위험률 등) 증가분(B)'이 훨씬 더 큰 폭으로 보험료를 상승시키는 주요 요인임을 알 수 있습니다. 이는 연령이 높아질수록 질병 발생률이나 사고 위험 등 보험사가 인지하는 위험도가 급격히 ...

✅ 2개의 Insight QA 생성 완료

📌 [INSIGHT_001] Insight
   Q: 이 보험 상품의 해지환급금 데이터를 분석했을 때, 조기 해지가 고객에게 미치는 재정적 영향은 무엇이며, 이는 상품 설계 철학에 대해 어떤 시사점을 제공합니까?
   A: 이 보험 상품을 조기에 해지할 경우, 고객은 상당한 재정적 손실을 입게 됩니다. 예를 들어, 1년 경과 후 해지 시 납입보험료 누계의 0%만 환급되며, 3년 후에도

## 5. Multi-Table QA 생성

복수의 테이블을 참조해야 답변 가능한 질문을 생성합니다.

In [22]:
# Multi-Table QA 생성
print("🔗 Multi-Table QA 생성 중...")

multi_table_qa = qa_generator.generate_multi_table_qa(
    tables=tables,
    num_questions=3
)

print(f"\n✅ {len(multi_table_qa)}개의 Multi-Table QA 생성 완료")
for qa in multi_table_qa:
    print(f"\n📌 [{qa.id}] {qa.difficulty}")
    print(f"   Q: {qa.question}")
    print(f"   A: {qa.answer}")
    if qa.evidence and 'required_tables' in str(qa.evidence):
        print(f"   참조 테이블: {qa.evidence}")

🔗 Multi-Table QA 생성 중...

✅ 3개의 Multi-Table QA 생성 완료

📌 [MULTI_001] Reasoning
   Q: table_2에서 10년 경과 시점의 납입보험료 누계가 6,000,000원일 때, 만약 이 보험이 20년납 상품이었다면 월 평균 보험료는 얼마였을까요? 그리고 이 월 평균 보험료는 table_4의 30세 남성 20년납 월보험료(45,000원)와 비교했을 때 얼마나 차이가 나나요?
   A: table_2의 10년 경과 시점 납입보험료 누계(6,000,000원)를 기준으로 20년납 상품의 월 평균 보험료를 계산하면 50,000원입니다. 이는 table_4의 30세 남성 20년납 월보험료(45,000원)보다 5,000원 더 높은 금액입니다.

📌 [MULTI_002] Calculation
   Q: table_4에서 40세 여성이 전기납 월보험료 상품에 가입했다고 할 때, 이 보험료가 table_1의 'XX세' 기준보험료(C)에 해당한다고 가정합니다. 이 경우, 'XX+2세' 시점의 기준보험료는 얼마로 예상할 수 있을까요? (단, table_1의 나이증가분(A)와 보험료 산출 기초율 증가분(B)는 table_1의 해당 값을 따릅니다.)
   A: table_4의 40세 여성 전기납 월보험료 47,000원을 'XX세' 기준보험료로 가정하면, 'XX+1세' 시점의 기준보험료는 47,000 + 1,059 + 10,846 = 58,905원입니다. 이어서 'XX+2세' 시점의 기준보험료는 58,905 + 1,357 + 13,897 = 74,159원입니다.

📌 [MULTI_003] Reasoning
   Q: 40세 남성이 20년납 월보험료 상품에 가입하여 5년 동안 보험료를 납입했을 경우, 총 납입한 보험료는 얼마인가요? 만약 이 시점에 일반암 진단을 받았다면, 총 납입 보험료 대비 암진단금은 몇 배에 해당하나요? 또한, 만약 이 보험의 해지환급금이 table_2의 '5년 경과' 시점의 납입보험료 누계(3,000,000원)

## 6. 꼬리 질문 (Follow-up) 생성

특정 QA에 대해 연속적인 후속 질문 체인을 생성합니다.

In [23]:
# 꼬리 질문 생성 (IR QA 기반)
if ir_qa_pairs:
    print("🔄 Follow-up QA 생성 중...")
    
    original_qa = ir_qa_pairs[0]
    print(f"\n원본 QA:")
    print(f"  Q: {original_qa.question}")
    print(f"  A: {original_qa.answer}")
    
    followup_result = qa_generator.generate_followup_qa(
        tables=tables,
        original_qa=original_qa
    )
    
    if followup_result and 'followup_chain' in followup_result:
        print(f"\n✅ {len(followup_result['followup_chain'])}개의 Follow-up QA 생성 완료")
        for i, followup in enumerate(followup_result['followup_chain'], 1):
            print(f"\n  🔹 Follow-up {i}:")
            print(f"     Q: {followup.get('question', 'N/A')}")
            print(f"     A: {followup.get('answer', 'N/A')}")
    else:
        print(f"\n결과: {followup_result}")

🔄 Follow-up QA 생성 중...

원본 QA:
  Q: XX세의 기준보험료는 얼마인가요?
  A: 42325

✅ 3개의 Follow-up QA 생성 완료

  🔹 Follow-up 1:
     Q: XX세에서 XX+1세로 나이가 증가할 때, 기준보험료는 얼마나 증가하나요?
     A: XX세의 기준보험료는 42325원이고, XX+1세의 기준보험료는 54321원이므로, 54321 - 42325 = 11996원 증가합니다.

  🔹 Follow-up 2:
     Q: 이 11996원의 기준보험료 증가분은 주로 '나이증가분(A)'과 '보험료 산출 기초율(위험률 등) 증가분(B)' 중 어떤 요인에 의해 발생한 것인가요?
     A: XX+1세의 '나이증가분(A)'은 1059원이고, '보험료 산출 기초율(위험률 등) 증가분(B)'은 10846원입니다. 두 요인 중 '보험료 산출 기초율(위험률 등) 증가분(B)'이 훨씬 크므로, 주로 이 요인에 의해 기준보험료가 증가했습니다.

  🔹 Follow-up 3:
     Q: '보험료 산출 기초율(위험률 등) 증가분(B)'이 '전년도 기준보험료의 최대 25% 가정'이라는 설명에 따르면, XX+1세의 B값(10846원)은 XX세의 기준보험료(42325원)의 25%를 초과하나요?
     A: XX세의 기준보험료 42325원의 25%는 42325 * 0.25 = 10581.25원입니다. XX+1세의 '보험료 산출 기초율 증가분(B)'은 10846원이므로, 10581.25원을 초과합니다.

✅ 3개의 Follow-up QA 생성 완료

  🔹 Follow-up 1:
     Q: XX세에서 XX+1세로 나이가 증가할 때, 기준보험료는 얼마나 증가하나요?
     A: XX세의 기준보험료는 42325원이고, XX+1세의 기준보험료는 54321원이므로, 54321 - 42325 = 11996원 증가합니다.

  🔹 Follow-up 2:
     Q: 이 11996원의 기준보험료 증가분은 주로 '나이증가분(A)'과 

## 7. Evol-Instruct: 질문 난이도 진화

기본 질문을 더 복잡한 질문으로 진화시킵니다.

In [24]:
# Evol-Instruct 적용
print("🧬 Evol-Instruct 적용 중...")

# 간단한 질문에서 시작
simple_question = "40세 남성의 20년납 월보험료는 얼마인가?"
print(f"\n원본 질문: {simple_question}")

evolved_result = qa_generator.evolve_question(
    tables=tables,
    original_question=simple_question
)

if evolved_result and 'evolved' in evolved_result:
    evolved = evolved_result['evolved']
    print(f"\n✅ 진화 완료")
    print(f"  진화 전략: {evolved.get('evolution_strategy', 'N/A')}")
    print(f"  진화된 질문: {evolved.get('question', 'N/A')}")
    print(f"  새로운 난이도: {evolved.get('difficulty', 'N/A')}")
    print(f"  답변: {evolved.get('answer', 'N/A')}")
else:
    print(f"\n결과: {evolved_result}")

🧬 Evol-Instruct 적용 중...

원본 질문: 40세 남성의 20년납 월보험료는 얼마인가?

✅ 진화 완료
  진화 전략: 제약 조건 추가 (Adding Constraints), 심층 추론 (Deepening Reasoning), 입력 복잡도 증가 (Complicating Input)
  진화된 질문: 40세 남성이 20년납으로 보험에 가입할 경우 월보험료는 65,000원이다. 만약 이 남성이 10년 뒤인 50세에 동일한 조건으로 가입한다면 월보험료는 얼마로 예상되는가? 그리고 table_1_premium_calculation 데이터를 참조하여 나이가 증가함에 따라 보험료가 상승하는 주요 원인 두 가지를 설명하시오.
  새로운 난이도: Reasoning (Level 5)
  답변: 50세에 동일한 조건으로 가입한다면 월보험료는 95,000원으로 예상됩니다. 보험료가 나이가 증가함에 따라 상승하는 주요 원인은 '나이증가분(A)'과 '보험료 산출 기초율(위험률 등) 증가분(B)'입니다. 이 두 가지 요소가 전년도 기준보험료에 더해져 새로운 기준보험료(C)를 형성하기 때문입니다.

✅ 진화 완료
  진화 전략: 제약 조건 추가 (Adding Constraints), 심층 추론 (Deepening Reasoning), 입력 복잡도 증가 (Complicating Input)
  진화된 질문: 40세 남성이 20년납으로 보험에 가입할 경우 월보험료는 65,000원이다. 만약 이 남성이 10년 뒤인 50세에 동일한 조건으로 가입한다면 월보험료는 얼마로 예상되는가? 그리고 table_1_premium_calculation 데이터를 참조하여 나이가 증가함에 따라 보험료가 상승하는 주요 원인 두 가지를 설명하시오.
  새로운 난이도: Reasoning (Level 5)
  답변: 50세에 동일한 조건으로 가입한다면 월보험료는 95,000원으로 예상됩니다. 보험료가 나이가 증가함에 따라 상승하는 주요 원인은 '나이증가분(A)'과 '보험료 산출 기초율(위험률 등) 증가분

## 8. LLM-as-Judge: 품질 평가

In [25]:
# QA 품질 평가
if ir_qa_pairs:
    print("⚖️ QA 품질 평가 중...")
    
    qa_to_evaluate = ir_qa_pairs[0]
    print(f"\n평가 대상 QA:")
    print(f"  Q: {qa_to_evaluate.question}")
    print(f"  A: {qa_to_evaluate.answer}")
    
    evaluation = qa_generator.evaluate_qa(
        tables=tables,
        qa_pair=qa_to_evaluate
    )
    
    if evaluation:
        print(f"\n✅ 평가 완료")
        print(f"  📊 Overall Score: {evaluation.overall_score}/5.0")
        print(f"  ✓ Pass: {'예' if evaluation.passed else '아니오'}")
        print(f"\n  세부 점수:")
        print(f"    - 정확성: {evaluation.correctness.get('score', 'N/A')}")
        print(f"    - 충실성: {evaluation.faithfulness.get('score', 'N/A')}")
        print(f"    - 관련성: {evaluation.relevance.get('score', 'N/A')}")
        print(f"    - 난이도 적절성: {evaluation.difficulty_appropriateness.get('score', 'N/A')}")
        print(f"    - 명확성: {evaluation.clarity.get('score', 'N/A')}")
        
        if evaluation.improvement_suggestions:
            print(f"\n  💡 개선 제안:")
            for suggestion in evaluation.improvement_suggestions:
                print(f"    - {suggestion}")

⚖️ QA 품질 평가 중...

평가 대상 QA:
  Q: XX세의 기준보험료는 얼마인가요?
  A: 42325

✅ 평가 완료
  📊 Overall Score: 5.0/5.0
  ✓ Pass: 예

  세부 점수:
    - 정확성: 5
    - 충실성: 5
    - 관련성: 5
    - 난이도 적절성: 5
    - 명확성: 5

✅ 평가 완료
  📊 Overall Score: 5.0/5.0
  ✓ Pass: 예

  세부 점수:
    - 정확성: 5
    - 충실성: 5
    - 관련성: 5
    - 난이도 적절성: 5
    - 명확성: 5


## 9. 종합 QA 데이터셋 생성

모든 난이도의 QA를 한 번에 생성하고 데이터셋으로 저장합니다.

In [ ]:
# 종합 QA 데이터셋 생성
print("📦 종합 QA 데이터셋 생성 중...")
print("  (모든 난이도 + Follow-up + Evol-Instruct)")
print("  ⏳ 약 2-3분 소요될 수 있습니다...\n")

comprehensive_dataset = qa_generator.generate_comprehensive_qa_dataset(
    tables=tables,
    questions_per_difficulty=2,
    include_followup=True,
    include_evolution=True,
    evaluate_quality=False  # 평가는 시간이 오래 걸려서 선택적으로
)

print("\n" + "="*60)
print("📊 데이터셋 생성 결과")
print("="*60)
print(f"  총 QA 쌍: {comprehensive_dataset['metadata'].get('total_qa_pairs', 0)}개")
print(f"  Follow-up 체인: {comprehensive_dataset['metadata'].get('total_followups', 0)}개")
print(f"  진화된 질문: {comprehensive_dataset['metadata'].get('total_evolved', 0)}개")

In [ ]:
# 생성된 QA 미리보기
print("\n📋 생성된 QA 샘플 (난이도별 1개씩):")
print("="*60)

shown_difficulties = set()
for qa in comprehensive_dataset['qa_pairs']:
    difficulty = qa.get('difficulty', 'Unknown')
    if difficulty not in shown_difficulties:
        shown_difficulties.add(difficulty)
        print(f"\n🔹 [{difficulty}]")
        print(f"   Q: {qa.get('question', 'N/A')}")
        answer = qa.get('answer', 'N/A')
        if len(str(answer)) > 150:
            print(f"   A: {str(answer)[:150]}...")
        else:
            print(f"   A: {answer}")

## 10. 데이터셋 저장

In [ ]:
# 출력 디렉토리 생성
output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)

# 타임스탬프 생성
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# JSON 파일로 저장
output_file = output_dir / f"insurance_qa_dataset_{timestamp}.json"

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(comprehensive_dataset, f, ensure_ascii=False, indent=2)

print(f"✅ 데이터셋 저장 완료: {output_file}")
print(f"   파일 크기: {output_file.stat().st_size / 1024:.1f} KB")

In [ ]:
# 간단한 통계 출력용 함수
def print_dataset_statistics(dataset: dict):
    """데이터셋 통계 출력"""
    print("\n" + "="*60)
    print("📈 QA 데이터셋 통계")
    print("="*60)
    
    # 난이도별 카운트
    difficulty_counts = {}
    answer_type_counts = {}
    
    for qa in dataset.get('qa_pairs', []):
        diff = qa.get('difficulty', 'Unknown')
        atype = qa.get('answer_type', 'Unknown')
        
        difficulty_counts[diff] = difficulty_counts.get(diff, 0) + 1
        answer_type_counts[atype] = answer_type_counts.get(atype, 0) + 1
    
    print("\n📊 난이도별 분포:")
    for diff, count in sorted(difficulty_counts.items()):
        bar = "█" * count
        print(f"   {diff:15} | {bar} ({count})")
    
    print("\n📊 답변 유형별 분포:")
    for atype, count in sorted(answer_type_counts.items()):
        bar = "█" * count
        print(f"   {atype:15} | {bar} ({count})")
    
    print(f"\n📊 총계:")
    print(f"   총 QA 쌍: {len(dataset.get('qa_pairs', []))}개")
    print(f"   Follow-up 체인: {len(dataset.get('followup_chains', []))}개")
    print(f"   진화된 질문: {len(dataset.get('evolved_questions', []))}개")

# 통계 출력
print_dataset_statistics(comprehensive_dataset)

## 11. 비동기 대량 생성 (선택사항)

더 빠른 생성을 위해 비동기 방식을 사용할 수 있습니다.

In [ ]:
# 비동기 QA 생성 (선택사항)
async def generate_async_dataset():
    """비동기 방식으로 QA 데이터셋 생성"""
    print("🚀 비동기 QA 데이터셋 생성 중...")
    
    dataset = await qa_generator.agenerate_comprehensive_qa_dataset(
        tables=tables,
        questions_per_difficulty=2
    )
    
    return dataset

# Jupyter에서 비동기 실행
# async_dataset = await generate_async_dataset()
# print(f"비동기 생성 완료: {len(async_dataset['qa_pairs'])}개 QA")

## 12. 커스텀 프롬프트로 특화 QA 생성

특정 요구사항에 맞는 커스텀 QA를 생성합니다.

In [ ]:
# 커스텀 프롬프트 예시: 수치 계산 중심 QA
CUSTOM_CALCULATION_PROMPT = """
## Task: 수치 계산 중심 QA 생성
보험 테이블에서 수치 계산이 필요한 질문을 생성하세요.

### Input Tables
{tables}

### Requirements
1. 반드시 수치 계산(사칙연산, 비율, 증감률 등)이 필요한 질문
2. 답변은 정확한 숫자로 제공
3. 계산 과정(Python 코드) 포함
4. 검증 가능한 형태로 출력

### Output Format (JSON)
```json
{{
    "questions": [
        {{
            "id": "CALC_001",
            "difficulty": "Aggregation",
            "answer_type": "calculation",
            "question": "질문",
            "answer": "수치 답변",
            "calculation_steps": ["단계1", "단계2"],
            "python_code": "검증용 Python 코드"
        }}
    ]
}}
```

### Generate 3 calculation-focused QA pairs.
"""

# 커스텀 프롬프트로 생성
from QA_example.prompts import format_tables_for_prompt

formatted_tables = format_tables_for_prompt(tables)
custom_prompt = CUSTOM_CALCULATION_PROMPT.format(tables=formatted_tables)

print("🔢 수치 계산 중심 QA 생성 중...")
response = qa_generator.pool.generate_content(custom_prompt)

# 결과 파싱
result = qa_generator._parse_json_response(response)

if 'questions' in result:
    print(f"\n✅ {len(result['questions'])}개의 계산 QA 생성 완료")
    for qa in result['questions']:
        print(f"\n📌 [{qa.get('id', 'N/A')}]")
        print(f"   Q: {qa.get('question', 'N/A')}")
        print(f"   A: {qa.get('answer', 'N/A')}")
        if 'python_code' in qa:
            print(f"   Python Code: 포함됨 ✅")
else:
    print(f"결과: {result}")

## 13. 요약 및 결론

### 생성된 QA 데이터셋의 특징

| 특징 | 설명 | 커버되는 양상 |
|------|------|-------------|
| 난이도 다양성 | IR부터 Insight까지 6단계 | #2 |
| Multi-Table | 복수 테이블 참조 필요 | #1 |
| 답변 유형 | Exact Match, Descriptive, Calculation | #3 |
| 수치 계산 | 집계, 비교, 증감률 계산 | #4, #6 |
| 꼬리 질문 | Follow-up 체인 생성 | #5 |
| 특정 셀 QA | 단일 셀 기반 Q-A | #7 |
| Evol-Instruct | 질문 난이도 진화 | #2, #4 |
| LLM Judge | 품질 평가 | #3 |